[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thedatasense/Patient_Early_Deterioration_Risk_Prediction/blob/master/notebooks/train_colab.ipynb)

# ICU Patient Deterioration Risk Prediction

This notebook trains a multimodal deep learning model for predicting patient deterioration in the ICU using MIMIC-IV data.

**Model Architecture:**
- BiLSTM temporal encoder for vital signs and lab values (48-hour sequences)
- Pre-computed ClinicalBERT embeddings for clinical notes
- Cross-modal attention fusion
- Binary classifier for 24-hour deterioration prediction

**Dataset:**
- 5.7M hourly prediction samples from 74,822 ICU stays
- 10 vital signs + 16 lab values
- ~2.8% positive rate (critical events within 24 hours)

**GPU Recommendations:**
| GPU | Speed | Best For |
|-----|-------|----------|
| **A100 40GB** | ~3-5 min/epoch | Full training (recommended) |
| T4 16GB | ~15-20 min/epoch | Testing / subset |
| A100 80GB | ~3-5 min/epoch | Overkill for this model |

**Requirements:**
- GPU runtime (A100 40GB recommended for full dataset)
- Google Drive with uploaded data files

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/thedatasense/Patient_Early_Deterioration_Risk_Prediction.git
%cd Patient_Early_Deterioration_Risk_Prediction

# Install dependencies
!pip install -q torch transformers pandas numpy pyarrow scikit-learn pyyaml tqdm matplotlib seaborn

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Data from Google Drive

**Before running:** Upload these files to your Google Drive:
- `samples_full.zip` (359 MB) - Full preprocessed dataset (5.7M samples)
- `embeddings.zip` (142 MB) - ClinicalBERT embeddings

Alternatively for faster training, use the subset:
- `samples_colab.zip` (74 MB) - Subset (280K samples)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration - UPDATE THIS PATH to match your Drive folder
DRIVE_DATA_PATH = "/content/drive/MyDrive/icu_deterioration_data"

# Choose dataset: "full" (5.7M samples) or "subset" (280K samples)
DATASET_TYPE = "full"  # Change to "subset" for faster experimentation

import os
os.makedirs("outputs/samples", exist_ok=True)
os.makedirs("outputs/embeddings", exist_ok=True)

In [ ]:
# Extract data from Drive
import zipfile
import shutil

# Select appropriate samples file
if DATASET_TYPE == "full":
    samples_zip = f"{DRIVE_DATA_PATH}/samples_full.zip"
else:
    samples_zip = f"{DRIVE_DATA_PATH}/samples_colab.zip"

embeddings_zip = f"{DRIVE_DATA_PATH}/embeddings.zip"

print(f"Extracting samples from {samples_zip}...")
with zipfile.ZipFile(samples_zip, 'r') as zip_ref:
    zip_ref.extractall("outputs/")

# Handle folder structure
if os.path.exists("outputs/samples_colab"):
    for f in os.listdir("outputs/samples_colab"):
        shutil.move(f"outputs/samples_colab/{f}", f"outputs/samples/{f}")
    os.rmdir("outputs/samples_colab")

print(f"Extracting embeddings from {embeddings_zip}...")
with zipfile.ZipFile(embeddings_zip, 'r') as zip_ref:
    zip_ref.extractall("outputs/embeddings/")

print("\nExtracted files:")
!ls -la outputs/samples/
!ls -la outputs/embeddings/

## 3. Load Data and Create Model

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from src.data.dataset import ICUDataset, create_dataloaders
from src.models.classifier import create_model
from src.training.losses import create_loss

# Configuration
SAMPLES_DIR = Path("outputs/samples")
EMBEDDINGS_DIR = Path("outputs/embeddings")
OUTPUT_DIR = Path("outputs/models/multimodal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 256 if DATASET_TYPE == "full" else 128  # Larger batch for full dataset
EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Dataset: {DATASET_TYPE}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Create data loaders
print("Creating data loaders...")
train_loader, val_loader, test_loader = create_dataloaders(
    samples_dir=SAMPLES_DIR,
    batch_size=BATCH_SIZE,
    num_workers=2,
    embeddings_dir=EMBEDDINGS_DIR,
)

# Get feature dimensions
train_dataset = train_loader.dataset
n_vitals = train_dataset.n_vitals
n_labs = train_dataset.n_labs

print(f"\nFeatures: {n_vitals} vitals, {n_labs} labs")
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_loader.dataset):,}")
print(f"Test samples: {len(test_loader.dataset):,}")
print(f"\nBatches per epoch: {len(train_loader):,}")

In [ ]:
# Model configuration
config = {
    "model": {
        "temporal_encoder": "lstm",
        "lstm": {"hidden_dim": 128, "num_layers": 2, "dropout": 0.3},
        "text": {"projection_dim": 256, "dropout": 0.1},
        "fusion": {"hidden_dim": 128},
        "classifier": {"hidden_dim": 64, "dropout": 0.3},
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-5,
        "max_epochs": EPOCHS,
        "patience": 10,
        "gradient_clip": 1.0,
        "use_amp": True,
        "loss": "focal",
        "focal_loss": {"alpha": 0.25, "gamma": 2.0},
    },
}

# Create model
print("Creating model...")
model = create_model(
    model_type="multimodal",
    n_vitals=n_vitals,
    n_labs=n_labs,
    config=config,
)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 4. Training

In [ ]:
import time
import json
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

# Create loss and optimizer
criterion = create_loss("focal", config)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# AMP scaler for mixed precision
scaler = torch.cuda.amp.GradScaler() if DEVICE == "cuda" else None

# Training tracking
best_val_auroc = 0.0
epochs_without_improvement = 0
history = []

print("Training setup complete!")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with progress bar."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for batch in pbar:
        vitals = batch["vitals"].to(device)
        labs = batch["labs"].to(device)
        mask = batch["mask"].to(device)
        static = batch["static"].to(device)
        embedding = batch["embedding"].to(device)
        has_notes = batch["has_notes"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
                loss = criterion(logits.squeeze(), labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
            loss = criterion(logits.squeeze(), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({"loss": f"{total_loss/n_batches:.4f}"})

    return total_loss / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate model."""
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch in tqdm(loader, desc="Validating", leave=False):
        vitals = batch["vitals"].to(device)
        labs = batch["labs"].to(device)
        mask = batch["mask"].to(device)
        static = batch["static"].to(device)
        embedding = batch["embedding"].to(device)
        has_notes = batch["has_notes"].to(device)
        labels = batch["label"].to(device)

        logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
        loss = criterion(logits.squeeze(), labels)

        total_loss += loss.item()
        probs = torch.sigmoid(logits.squeeze()).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader)
    auroc = roc_auc_score(all_labels, all_probs)
    auprc = average_precision_score(all_labels, all_probs)

    return avg_loss, auroc, auprc

In [ ]:
# Training loop
PATIENCE = 10

print(f"Starting training for {EPOCHS} epochs...")
print(f"Training on {len(train_loader.dataset):,} samples")
print()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)

    # Validate
    val_loss, val_auroc, val_auprc = validate(model, val_loader, criterion, DEVICE)

    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - epoch_start

    # Record history
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auroc": val_auroc,
        "val_auprc": val_auprc,
        "lr": current_lr,
        "time": epoch_time,
    })

    # Print progress
    improvement = "✓ NEW BEST" if val_auroc > best_val_auroc else ""
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
          f"AUROC: {val_auroc:.4f} | "
          f"AUPRC: {val_auprc:.4f} | "
          f"{epoch_time:.0f}s {improvement}")

    # Check for improvement
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        epochs_without_improvement = 0
        # Save best model
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auroc": best_val_auroc,
            "config": config,
        }, OUTPUT_DIR / "best_model.pt")
    else:
        epochs_without_improvement += 1

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*50}")
print(f"Training complete! Best Val AUROC: {best_val_auroc:.4f}")
print(f"{'='*50}")

## 5. Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(OUTPUT_DIR / "best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

# Evaluate on test set
test_loss, test_auroc, test_auprc = validate(model, test_loader, criterion, DEVICE)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"AUROC: {test_auroc:.4f}")
print(f"AUPRC: {test_auprc:.4f}")
print(f"Loss:  {test_loss:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = [h["epoch"] for h in history]

# Loss
axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train", linewidth=2)
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUROC
axes[1].plot(epochs_range, [h["val_auroc"] for h in history], linewidth=2, color='green')
axes[1].axhline(y=test_auroc, color='r', linestyle='--', label=f"Test: {test_auroc:.4f}")
axes[1].axhline(y=best_val_auroc, color='g', linestyle=':', alpha=0.5, label=f"Best Val: {best_val_auroc:.4f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUROC")
axes[1].set_title("Validation AUROC")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUPRC
axes[2].plot(epochs_range, [h["val_auprc"] for h in history], linewidth=2, color='orange')
axes[2].axhline(y=test_auprc, color='r', linestyle='--', label=f"Test: {test_auprc:.4f}")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUPRC")
axes[2].set_title("Validation AUPRC")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed classification report
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions on test set
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Getting predictions"):
        vitals = batch["vitals"].to(DEVICE)
        labs = batch["labs"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        static = batch["static"].to(DEVICE)
        embedding = batch["embedding"].to(DEVICE)
        has_notes = batch["has_notes"].to(DEVICE)

        logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
        probs = torch.sigmoid(logits.squeeze()).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(batch["label"].numpy().tolist())

# Find optimal threshold
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5

print(f"Optimal threshold: {optimal_threshold:.3f}")

# Classification report
preds = (np.array(all_probs) >= optimal_threshold).astype(int)
print("\nClassification Report:")
print(classification_report(all_labels, preds, target_names=["No Event", "Deterioration"]))

# Confusion matrix
cm = confusion_matrix(all_labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["No Event", "Deterioration"],
            yticklabels=["No Event", "Deterioration"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix (threshold={optimal_threshold:.3f})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Save results
results = {
    "dataset_type": DATASET_TYPE,
    "train_samples": len(train_loader.dataset),
    "val_samples": len(val_loader.dataset),
    "test_samples": len(test_loader.dataset),
    "best_val_auroc": float(best_val_auroc),
    "test_auroc": float(test_auroc),
    "test_auprc": float(test_auprc),
    "test_loss": float(test_loss),
    "optimal_threshold": float(optimal_threshold),
    "epochs_trained": len(history),
    "model_params": n_params,
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR}")

## 6. Download Results

In [ ]:
# Copy results to Drive for persistence
import shutil

drive_output = f"{DRIVE_DATA_PATH}/model_outputs"
os.makedirs(drive_output, exist_ok=True)

# Copy model and results
for f in ["best_model.pt", "results.json", "training_curves.png", "confusion_matrix.png"]:
    src = OUTPUT_DIR / f
    if src.exists():
        shutil.copy(src, f"{drive_output}/{f}")
        print(f"Copied {f} to Drive")

print(f"\nResults saved to: {drive_output}")

In [ ]:
# Alternative: Download directly
from google.colab import files

!zip -r model_outputs.zip outputs/models/
files.download("model_outputs.zip")